In [2]:
from okx import OrderbookStore
import polars as pl
from datetime import date, datetime

store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

In [3]:
dates = [date(2025, 9, 2)]
df = store.get(
    inst_type='FUTURES',
    inst_family='BTC-USD',
    dates=dates,
    depth=1,
    features=['trim']
).collect()
print(f"Shape: {df.shape}")
print(df.head())

mb_size = df.estimated_size("mb") if hasattr(df, "estimated_size") else df.estimated_size() / (1024 * 1024)
print(f"Size: {mb_size:.2f} MB")


Shape: (14793203, 9)
shape: (5, 9)
┌────────────┬───────────┬───────────┬──────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ timeMs     ┆ exchTimeM ┆ symbol    ┆ bid_1_px ┆ … ┆ bid_1_ord ┆ ask_1_px ┆ ask_1_qty ┆ ask_1_ord │
│ ---        ┆ s         ┆ ---       ┆ ---      ┆   ┆ Cnt       ┆ ---      ┆ ---       ┆ Cnt       │
│ i64        ┆ ---       ┆ str       ┆ f64      ┆   ┆ ---       ┆ f64      ┆ f64       ┆ ---       │
│            ┆ i64       ┆           ┆          ┆   ┆ i32       ┆          ┆           ┆ i32       │
╞════════════╪═══════════╪═══════════╪══════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ 1756771200 ┆ 175677120 ┆ BTC-USD-2 ┆ 109291.4 ┆ … ┆ 1         ┆ 109297.3 ┆ 2.0       ┆ 1         │
│ 055        ┆ 0053      ┆ 50905.OK  ┆          ┆   ┆           ┆          ┆           ┆           │
│ 1756771200 ┆ 175677120 ┆ BTC-USD-2 ┆ 109291.4 ┆ … ┆ 1         ┆ 109297.3 ┆ 2.0       ┆ 1         │
│ 065        ┆ 0063      ┆ 50905.OK  ┆          ┆   ┆   

In [3]:
bin_freq = '3s'

from okx.features import bin_ob
import time

def bench_bin_ob_proper(df, bin_freq, ff, count, n_runs=5):
    mode = f"ff={ff}, count={count}"
    times = []
    
    for _ in range(n_runs):
        # Force re-planning by recreating the lazy query each time
        lf = df.lazy() if hasattr(df, 'lazy') else df
        
        start = time.perf_counter()
        result = bin_ob(lf, bin_freq, ff=ff, count=count).collect()
        end = time.perf_counter()
        
        times.append(end - start)
    
    # Report median and min (min shows best-case performance)
    times_sorted = sorted(times)
    median_time = times_sorted[len(times_sorted) // 2]
    min_time = times_sorted[0]
    
    print(f"({mode}) Median: {median_time:.4f}s, Min: {min_time:.4f}s, All: {[f'{t:.4f}' for t in times]}")
    return result

# Run benchmarks
results = {}
for ff in [False, True]:
    for count in [False, True]:
        key = (ff, count)
        results[key] = bench_bin_ob_proper(df, bin_freq, ff=ff, count=count)

# For compatibility with later notebook cells
df_binned_ff = results[(True, False)]
df_binned_count = results[(False, True)]
df_binned_with_count = df_binned_count
df_binned = results[(False, False)]



(ff=False, count=False) Median: 0.4015s, Min: 0.3860s, All: ['0.4720', '0.4015', '0.3951', '0.4022', '0.3860']
(ff=False, count=True) Median: 0.3784s, Min: 0.3733s, All: ['0.3846', '0.3733', '0.3784', '0.3796', '0.3774']
(ff=True, count=False) Median: 0.3909s, Min: 0.3829s, All: ['0.3935', '0.3872', '0.3829', '0.3914', '0.3909']
(ff=True, count=True) Median: 0.3915s, Min: 0.3852s, All: ['0.3937', '0.3911', '0.3852', '0.3964', '0.3915']


In [4]:
from okx.features import build_flush_features
start = datetime.now()
FLUSH_FEATURES = build_flush_features(df, binning=None, inst_type='FUTURES')
end = datetime.now()
print(f"Time taken: {end - start}")


Time taken: 0:00:00.000048
